# Clasificación de sentimiento en TASS con características léxicas y TF-IDF

**Curso de Procesamiento de Lenguaje Natural ** · Unidad: Text Classification

Este notebook desarrolla el modelo de clasificación de polaridad sobre **TASS 2018** (tweets en español; clases `P`, `N`, `NEU`, `NONE`) usando los módulos de apoyo del curso: `text_processing.py`, `feature_extraction.py` y `lexical_features.py`.

**Pregunta de investigación.** ¿Aportan las características léxicas diseñadas manualmente (29 rasgos: pronombres, adverbios, adjetivos, etiquetas de Twitter, estadísticos de longitud de palabra) información que TF-IDF no captura? ¿Y cuánto cambia la respuesta según el preprocesamiento aplicado?

**Diseño experimental.**

| Enfoque | Representación | Clasificador |
|---|---|---|
| Baseline | ninguna (clase mayoritaria) | Dummy |
| Solo léxico | 29 rasgos de `FeatureExtraction` | Regresión logística / Random Forest |
| Solo TF-IDF | unigramas + bigramas | Regresión logística |
| Híbrido | TF-IDF + 29 rasgos léxicos | Regresión logística |

Cada enfoque léxico se evalúa con **dos preprocesamientos** (el original de `TextProcessing.transformer` y una variante que conserva acentos y etiquetas), porque el diagnóstico de la sección 4 muestra que el original desactiva parte de los rasgos.

**Referencias.** Jurafsky & Martin, *Speech and Language Processing* (borrador 2026): cap. 4 (Logistic Regression and Text Classification) y Apéndice B (Naive Bayes).

**Criterios de evaluación.** Métrica principal: macro-F1 (las clases están desbalanceadas). Se reporta también accuracy. La selección de modelos usa validación cruzada sobre *train*; *test* se usa una sola vez al final.

## 1. Configuración

Coloca este notebook en la misma carpeta que `text_processing.py`, `feature_extraction.py`, `lexical_features.py` y `utils.py`.

`feature_extraction.py` importa `utils.Utils`, un archivo que no venía entre los adjuntos. El `utils.py` entregado junto a este notebook es un stub mínimo (solo imprime el traceback).

Dependencias: `pip install scikit-learn pandas numpy scipy matplotlib nltk spacy`.

In [1]:
import sys, re, unicodedata, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))

from logic.text_processing import TextProcessing
from logic.feature_extraction import FeatureExtraction
from logic.lexical_features import lexical_es

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

import sklearn
print("Python", platform.python_version(), "| sklearn", sklearn.__version__, "| pandas", pd.__version__)

Python 3.14.6 | sklearn 1.9.0 | pandas 3.0.3


## 2. Datos: TASS 2018

Se usan **exclusivamente** los CSV reales de TASS 2018. Si no se encuentran, el notebook se detiene con un error que indica las rutas revisadas y los CSV disponibles; no existen datos de respaldo.

In [3]:
DIR_DATA = Path.cwd() / 'data' / 'tass'
PATH = 'data/tass'
print(f'Data directory: {DIR_DATA}')

Data directory: C:\Users\epuerta\OneDrive - Universidad Tecnológica de Bolívar\Apps\courseNLP\examples\data\tass


In [7]:
# --- Rutas y columnas (ajusta si tus archivos se llaman distinto) ---------------------------------
DIR_CANDIDATES = [Path(DIR_DATA), Path(PATH),]
TRAIN_FILE = "tass2018_es_train.csv"
TEST_FILE  = "tass2018_es_test.csv"
COL_TEXT   = "content"
COL_LABEL  = "sentiment/polarity/value"
LABELS     = ["N", "NEU", "NONE", "P"]

def read_csv_flexible(path):
    return pd.read_csv(path, sep=None, engine="python", encoding="utf-8")


def load_tass():
    for d in DIR_CANDIDATES:
        tr, te = d / TRAIN_FILE, d / TEST_FILE
        if tr.exists():
            df_tr = read_csv_flexible(tr)
            df_te = read_csv_flexible(te) if te.exists() else None
            return df_tr, df_te, f"TASS real ({d})"
    found = [str(p) for d in DIR_CANDIDATES if d.exists() for p in d.glob("*.csv")]
    raise FileNotFoundError(
        f"No se encontró {TRAIN_FILE}. Rutas revisadas: {[str(d) for d in DIR_CANDIDATES]}. "
        f"CSV disponibles: {found}. Ajusta DIR_CANDIDATES, TRAIN_FILE y TEST_FILE.")


df_train, df_test, DATA_SOURCE = load_tass()
print("Fuente de datos:", DATA_SOURCE)
faltan = [c for c in (COL_TEXT, COL_LABEL) if c not in df_train.columns]
if faltan:
    raise KeyError(f"Faltan las columnas {faltan}. Columnas disponibles: {df_train.columns.tolist()}")

# Normalización de etiquetas y validación de test
df_train[COL_LABEL] = df_train[COL_LABEL].astype(str).str.strip().str.upper()
if df_test is not None:
    df_test[COL_LABEL] = df_test[COL_LABEL].astype(str).str.strip().str.upper()
    if not df_test[COL_LABEL].isin(LABELS).any():          # test sin etiquetas públicas
        print("El archivo de test no trae etiquetas válidas: se usa una partición estratificada de train.")
        df_test = None
if df_test is None:
    df_train, df_test = train_test_split(df_train, test_size=0.2, stratify=df_train[COL_LABEL], random_state=SEED)

df_train = df_train.dropna(subset=[COL_TEXT]).reset_index(drop=True)
df_test  = df_test.dropna(subset=[COL_TEXT]).reset_index(drop=True)

dist = pd.DataFrame({"train": df_train[COL_LABEL].value_counts(),
                     "test": df_test[COL_LABEL].value_counts()}).reindex(LABELS)
dist["train_%"] = (100 * dist["train"] / dist["train"].sum()).round(1)
print(f"\ntrain={len(df_train)}  test={len(df_test)}")
dist

Fuente de datos: TASS real (C:\Users\epuerta\OneDrive - Universidad Tecnológica de Bolívar\Apps\courseNLP\examples\data\tass)

train=1008  test=506


,train,test,train_%
sentiment/polarity/value,,,
N,418,219,41.5
NEU,133,69,13.2
NONE,139,62,13.8
P,318,156,31.5


## 3. Preprocesamiento y adaptadores

Dos versiones del texto:

- **`clean` (original):** `TextProcessing.transformer(texto)`. Se usa para TF-IDF y como primer preprocesamiento para los rasgos léxicos. Sin eliminación de *stopwords*: en sentimiento, palabras como *no* o *nunca* portan polaridad.
- **`lex_text` (variante):** conserva acentos y sustituye URL, menciones, hashtags y emojis por las etiquetas que `FeatureExtraction` sí cuenta (`url`, `mention`, `hashtag`, `emoji`). Reutiliza `TextProcessing.remove_patterns`.

**Adaptador `LexicalVectorizer`.** `FeatureExtraction.transform()` no sirve directamente en un `Pipeline` de scikit-learn: recibe una lista pero `get_features()` espera *un* mensaje, y además aplica `abs()` (pierde el signo de curtosis y asimetría) y falla con `None` cuando el mensaje queda vacío. El adaptador llama por mensaje a `get_features_lexical()` (que devuelve el vector sin `abs`) y usa un vector de ceros si el mensaje no produce rasgos.

In [8]:
FEATURE_NAMES = [
    "weighted_position", "weighted_normalized",
    "label_mention", "label_url", "label_hashtag", "label_emoji", "label_retweets",
    "lexical_diversity", "label_word",
    "first_person_singular", "second_person_singular", "third_person_singular",
    "first_person_plurar", "second_person_plurar", "third_person_plurar",
    "avg_word", "kur_word", "skew_word",
    "adverb_neg", "adverb_time", "adverb_place", "adverb_mode", "adverb_cant", "adverb_all",
    "adjetives_neg", "adjetives_pos", "who_general", "who_male", "who_female",
]

URL_RE   = re.compile(r"(?i)(?:https?://|www\.)\S+")
EMOJI_RE = re.compile("[\U0001F000-\U000E007F\u2600-\u27BF]")


def normalize_keep_accents(text: str) -> str:
    """Variante de TextProcessing.transformer: conserva acentos/ñ y emite las etiquetas que se cuentan."""
    t = unicodedata.normalize("NFC", str(text)).lower()
    t = URL_RE.sub(" url ", t)
    t = re.sub(r"@\w{1,40}", " mention ", t)
    t = re.sub(r"#\w{1,40}", " hashtag ", t)
    t = EMOJI_RE.sub(" emoji ", t)
    t = TextProcessing.remove_patterns(t)
    return re.sub(r"\s+", " ", t).strip()


def original_clean(text: str) -> str:
    return TextProcessing.transformer(str(text)) or ""


class LexicalVectorizer(BaseEstimator, TransformerMixin):
    """Envuelve FeatureExtraction.get_features_lexical: un vector de 29 rasgos por mensaje."""
    def __init__(self, lang="es"):
        self.lang = lang

    def fit(self, X, y=None):
        self.fe_ = FeatureExtraction(self.lang)
        return self

    def transform(self, X):
        if not hasattr(self, "fe_"):
            self.fit(X)
        rows = []
        for msg in np.asarray(X, dtype=object).ravel():
            try:
                v = self.fe_.get_features_lexical(msg if isinstance(msg, str) else "")
                v = None if v is None else np.asarray(v, dtype=np.float64)
            except Exception:
                v = None
            if v is None or v.shape[0] != len(FEATURE_NAMES) or not np.isfinite(v).all():
                v = np.zeros(len(FEATURE_NAMES))
            rows.append(v)
        return np.vstack(rows)


def build_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Tabla de características. LexicalVectorizer no aprende nada de los datos: calcularlo antes del CV no filtra información."""
    raw = df[COL_TEXT].astype(str)
    out = pd.DataFrame({"raw": raw.values,
                        "clean": raw.map(original_clean).values,
                        "lex_text": raw.map(normalize_keep_accents).values})
    lv = LexicalVectorizer().fit(out["clean"])
    m1, m2 = lv.transform(out["clean"]), lv.transform(out["lex_text"])
    for i, name in enumerate(FEATURE_NAMES):
        out["v1_" + name] = m1[:, i]     # preprocesamiento original
        out["v2_" + name] = m2[:, i]     # variante con acentos y etiquetas
    return out


Xtr, Xte = build_frame(df_train), build_frame(df_test)
ytr, yte = df_train[COL_LABEL].values, df_test[COL_LABEL].values
V1 = ["v1_" + n for n in FEATURE_NAMES]
V2 = ["v2_" + n for n in FEATURE_NAMES]
print("Matriz de rasgos por mensaje:", len(FEATURE_NAMES), "columnas léxicas × 2 variantes")
Xtr[["raw", "clean", "lex_text"]].head(6)

Matriz de rasgos por mensaje: 29 columnas léxicas × 2 variantes


,raw,clean,lex_text
0,-Me caes muy bien \r\n-Tienes que jugar más partidas al lol con Russel y conmigo\r\n-P...,me caes muy bien tienes que jugar mas partidas al lol con russel y conmigo por que tan...,me caes muy bien tienes que jugar más partidas al lol con russel y conmigo por qué tan...
1,@myendlesshazza a. que puto mal escribo\r\n\r\nb. me sigo surrando help \r\n\r\n3. ha ...,mention a. que puto mal escribo b. me sigo surrando help 3. ha quedado raro el cometel...,mention a. que puto mal escribo b. me sigo surrando help 3. ha quedado raro el cómetel...
2,@estherct209 jajajaja la tuya y la d mucha gente seguro!! Pero yo no puedo sin mi mele...,mention jajajaja la tuya y la d mucha gente seguro pero yo no puedo sin mi melena me m...,mention jajajaja la tuya y la d mucha gente seguro pero yo no puedo sin mi melena me m...
3,Quiero mogollón a @AlbaBenito99 pero sobretodo por lo rápido que contesta a los wasaps,quiero mogollon a mention pero sobretodo por lo rapido que contesta a los wasaps,quiero mogollón a mention pero sobretodo por lo rápido que contesta a los wasaps
4,Vale he visto la tia bebiendose su regla y me hs dado muchs grima,vale he visto la tia bebiendose su regla y me hs dado muchs grima,vale he visto la tia bebiendose su regla y me hs dado muchs grima
5,@Yulian_Poe @guillermoterry1 Ah. mucho más por supuesto! solo que lo incluyo. Me había...,mention mention ah. mucho mas por supuesto solo que lo incluyo. me habias entendido mal,mention mention ah. mucho más por supuesto solo que lo incluyo. me habías entendido mal


## 4. Diagnóstico de los módulos de apoyo

Antes de entrenar, se verifica qué hace realmente cada etapa. Estos hallazgos condicionan la interpretación de cualquier resultado posterior.

In [9]:
def elegir_tweet_real(textos):
    """Tweet real de train con más marcas a la vez: emoji, hashtag, acento/ñ, mención y URL."""
    marcas = [EMOJI_RE, re.compile(r"#\w"), re.compile("[áéíóúüñÁÉÍÓÚÜÑ]"), re.compile(r"@\w"), URL_RE]
    puntajes = [sum(bool(m.search(t)) for m in marcas) for t in textos]
    return textos[int(np.argmax(puntajes))]

ejemplo = elegir_tweet_real(df_train[COL_TEXT].astype(str).tolist())      # tweet real, no escrito a mano
t1, t2 = original_clean(ejemplo), normalize_keep_accents(ejemplo)
print("Original         :", ejemplo)
print("transformer()    :", t1)
print("keep_accents()   :", t2)

lv = LexicalVectorizer().fit([ejemplo])
v1 = pd.Series(lv.transform([t1])[0], index=FEATURE_NAMES)
v2 = pd.Series(lv.transform([t2])[0], index=FEATURE_NAMES)
cmp_ = pd.DataFrame({"original": v1, "con acentos/etiquetas": v2})
print("\nRasgos que cambian entre variantes:")
cmp_[cmp_["original"] != cmp_["con acentos/etiquetas"]]

Original         : Buenos días Magaluf !  👌🏻✅
• @BHmallorca • #BHmallorca#Magaluf #Mallorca #Island #Summer2016… https://t.co/amAnsSDaxI
transformer()    : buenos dias magaluf mention hastaghastag hastag hastag hastag url
keep_accents()   : buenos días magaluf emoji emoji emoji • mention • hashtag hashtag hashtag hashtag hashtag … url

Rasgos que cambian entre variantes:


,original,con acentos/etiquetas
weighted_position,2.894444,3.623214
weighted_normalized,4.666667,7.562500
label_hashtag,0.000000,5.000000
label_emoji,0.000000,3.000000
lexical_diversity,0.261500,0.221100
label_word,7.000000,6.000000
avg_word,6.714300,3.333300
kur_word,1.176800,-1.620500
skew_word,1.458700,0.326900


**Lo que muestra la celda anterior** (sobre un tweet real de *train*, elegido por reunir emoji, hashtag, acentos, mención y URL; si el elegido no tiene alguna de esas marcas, el efecto correspondiente no se observa en él y se verifica en la tabla de cobertura siguiente):

1. `transformer()` aplica `proper_encoding` (NFD → ASCII con `ignore`) **antes** de buscar emojis. El emoji desaparece y la etiqueta `[EMOJI]` nunca se genera, por lo que `label_emoji` vale siempre 0.
2. La etiqueta de hashtag se emite como `hastag` (errata), pero `FeatureExtraction` cuenta `hashtag`. `label_hashtag` vale siempre 0.
3. Se eliminan los acentos y la ñ (*más* → *mas*, *niño* → *nino*), pero los léxicos contienen entradas con acento. Esas entradas no pueden coincidir.

In [10]:
# Alcanzabilidad del léxico: entradas que ningún token puede igualar con cada preprocesamiento
cats = [k for k in lexical_es if k != "hate"]        # 'hate' no la usa FeatureExtraction
rows = []
for k in cats:
    entries = lexical_es[k]
    con_acento = [w for w in entries if w != TextProcessing.proper_encoding(w)]
    multipalabra = [w for w in entries if " " in w.strip()]
    rows.append({"categoría": k, "entradas": len(entries),
                 "inalcanzables_original": len(set(con_acento) | set(multipalabra)),
                 "inalcanzables_variante": len(set(multipalabra))})
reach = pd.DataFrame(rows).set_index("categoría")
reach.loc["TOTAL"] = reach.sum()
reach["% perdido (original)"] = (100 * reach["inalcanzables_original"] / reach["entradas"]).round(1)
display(reach)

solape = sorted(set(lexical_es["adjetives_neg"]) & set(lexical_es["adjetives_pos"]))
print(f"\nAdjetivos presentes a la vez en 'adjetives_neg' y 'adjetives_pos' ({len(solape)}):", solape)
ambiguos = [w for k in ["first_person_singular", "third_person_singular"] for w in lexical_es[k] if w in ("el", "mi", "tu", "ese")]
print("Entradas ambiguas (artículo/posesivo/demostrativo tratadas como pronombre personal):",
      sorted(set(ambiguos + [w for w in lexical_es["second_person_singular"] if w == "tu"])))

,entradas,inalcanzables_original,inalcanzables_variante,% perdido (original)
categoría,,,,
first_person_singular,3,0,0,0.0
second_person_singular,5,1,0,20.0
third_person_singular,6,1,0,16.7
first_person_plurar,2,0,0,0.0
second_person_plurar,5,0,0,0.0
third_person_plurar,6,0,0,0.0
adverb_time,12,4,0,33.3
adverb_neg,4,1,1,25.0
adverb_place,13,5,0,38.5



Adjetivos presentes a la vez en 'adjetives_neg' y 'adjetives_pos' (6): ['caliente', 'diferente', 'duro', 'imposible', 'popular', 'salado']
Entradas ambiguas (artículo/posesivo/demostrativo tratadas como pronombre personal): ['el', 'ese', 'mi', 'tu']


In [11]:
# Efecto sobre el corpus: cobertura de rasgos y redundancia de rasgos de posición
n_tokens = Xtr["clean"].map(lambda s: len(TextProcessing.tokenizer(s) or []))
resumen = pd.DataFrame({
    "% mensajes con valor > 0 (original)": (Xtr[V1] > 0).mean().values * 100,
    "% mensajes con valor > 0 (variante)": (Xtr[V2] > 0).mean().values * 100,
}, index=FEATURE_NAMES).round(1)
display(resumen.loc[["label_hashtag", "label_emoji", "first_person_singular", "second_person_singular",
                     "adverb_time", "adverb_place", "adverb_mode", "adjetives_neg", "adjetives_pos"]])

r_wp = np.corrcoef(Xtr["v1_weighted_position"], np.log1p(n_tokens))[0, 1]
r_wn = np.corrcoef(Xtr["v1_weighted_normalized"], n_tokens)[0, 1]
print(f"\ncorr(weighted_position, log(1+n_tokens))  = {r_wp:.3f}")
print(f"corr(weighted_normalized, n_tokens)       = {r_wn:.3f}")
zeros = int((Xtr[V1].abs().sum(axis=1) == 0).sum())
print(f"Mensajes con vector léxico vacío (original): {zeros} de {len(Xtr)}")

,% mensajes con valor > 0 (original),% mensajes con valor > 0 (variante)
label_hashtag,0.0,7.1
label_emoji,0.0,3.3
first_person_singular,17.9,17.3
second_person_singular,4.9,4.9
adverb_time,18.0,18.7
adverb_place,1.4,3.4
adverb_mode,10.0,12.5
adjetives_neg,13.0,13.3
adjetives_pos,21.7,22.2



corr(weighted_position, log(1+n_tokens))  = 0.733
corr(weighted_normalized, n_tokens)       = 0.984
Mensajes con vector léxico vacío (original): 0 de 1008


**Hallazgos adicionales (verificables en las celdas anteriores):**

- `weighted_position` es la suma de 1/(1+índice de la *primera* aparición de cada token); con tokens sin repetir equivale al número armónico H(n). `weighted_normalized` equivale a (n+1)/2. Ambos son, en la práctica, funciones de la longitud del mensaje y no de la posición de las palabras. Una correlación alta con la longitud (celda anterior) lo confirma.
- `lexical_diversity` calcula la proporción de **caracteres** distintos sobre el total de caracteres, no una razón tipo/token de palabras. Mide algo distinto a lo que sugiere su nombre.
- Las entradas multipalabra (`en absoluto`, `bestia negra`) no coinciden nunca porque la comparación es por token.
- Los léxicos contienen palabras no relacionadas con polaridad en `adjetives_neg`/`adjetives_pos` (colores, tamaños) y entradas en ambas listas. Es un léxico de categorías gramaticales/semánticas, no un léxico de sentimiento validado.
- `el` (artículo) está en `third_person_singular`, por lo que cada aparición del artículo suma como pronombre personal (lo mismo ocurre con `mi`, `tu` y `ese`).

**Consecuencia para el experimento:** el enfoque léxico puede compararse en dos condiciones (original y variante). La diferencia entre ambas cuantifica cuánto pesa el preprocesamiento sobre el rendimiento.

## 5. Experimentos

**Protocolo.**

- Validación cruzada estratificada de 5 particiones sobre *train*; métrica principal macro-F1.
- Desbalance tratado con `class_weight="balanced"` dentro del modelo. Así no se duplican ejemplos antes de la partición y se evita fuga de información entre particiones.
- TF-IDF ajustado dentro de cada partición (parte del `Pipeline`); usa `TextProcessing.tokenizer` (TweetTokenizer) como tokenizador.
- Rasgos léxicos estandarizados con `StandardScaler` dentro del `Pipeline`.
- *Test* se evalúa una sola vez, después de fijar los modelos.

In [13]:
def tfidf():
    return TfidfVectorizer(tokenizer=TextProcessing.tokenizer, token_pattern=None, lowercase=False,
                           ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def lr():
    return LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000, random_state=SEED)

def rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

def make_pipe(use_tfidf, lex_cols, clf):
    parts = []
    if use_tfidf: parts.append(("tfidf", tfidf(), "clean"))
    if lex_cols:  parts.append(("lex", StandardScaler(), lex_cols))
    return Pipeline([("feat", ColumnTransformer(parts, sparse_threshold=0.3)), ("clf", clf)])

EXPERIMENTS = {
    "0. Mayoritaria (baseline)":         Pipeline([("clf", DummyClassifier(strategy="most_frequent"))]),
    "1. Léxico (original) + LR":         make_pipe(False, V1, lr()),
    "2. Léxico (variante) + LR":         make_pipe(False, V2, lr()),
    "3. Léxico (variante) + RF":         make_pipe(False, V2, rf()),
    "4. TF-IDF + LR":                    make_pipe(True, None, lr()),
    "5. TF-IDF + léxico (original) + LR": make_pipe(True, V1, lr()),
    "6. TF-IDF + léxico (variante) + LR": make_pipe(True, V2, lr()),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, pipe in EXPERIMENTS.items():
    res = cross_validate(pipe, Xtr, ytr, cv=cv, scoring={"f1": "f1_macro", "acc": "accuracy"}, n_jobs=1)
    cv_rows.append({"experimento": name,
                    "CV macro-F1": res["test_f1"].mean(), "± std": res["test_f1"].std(),
                    "CV accuracy": res["test_acc"].mean()})
cv_table = pd.DataFrame(cv_rows).set_index("experimento").round(4)
print("Fuente de datos:", DATA_SOURCE)
cv_table

TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
# Evaluación final en test (una sola vez) y almacenamiento de predicciones
fitted, preds, test_rows = {}, {}, []
for name, pipe in EXPERIMENTS.items():
    pipe.fit(Xtr, ytr)
    p = pipe.predict(Xte)
    fitted[name], preds[name] = pipe, p
    test_rows.append({"experimento": name,
                      "test macro-F1": f1_score(yte, p, average="macro", labels=LABELS),
                      "test accuracy": accuracy_score(yte, p)})
test_table = pd.DataFrame(test_rows).set_index("experimento").round(4)
final = cv_table[["CV macro-F1", "± std"]].join(test_table)
print("Fuente de datos:", DATA_SOURCE)
final

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(final))
ax.barh(y - 0.2, final["CV macro-F1"], height=0.38, xerr=final["± std"], label="CV (train)", color="#2C4293")
ax.barh(y + 0.2, final["test macro-F1"], height=0.38, label="Test", color="#098559")
ax.set_yticks(y); ax.set_yticklabels(final.index); ax.invert_yaxis()
ax.set_xlabel("macro-F1"); ax.legend(loc="lower right")
ax.set_title("Comparación de enfoques · TASS 2018")
plt.tight_layout(); plt.show()

## 6. Análisis

### 6.1 ¿La diferencia es real o ruido?

Con conjuntos de test pequeños, diferencias de uno o dos puntos de macro-F1 pueden ser ruido de muestreo. Se estima el intervalo de la diferencia con *bootstrap* pareado sobre *test* (mismas muestras remuestreadas para ambos modelos).

In [14]:
def paired_bootstrap(y_true, p_a, p_b, n=1000, seed=SEED):
    """Diferencia de macro-F1 (b - a) con IC 95% por bootstrap pareado."""
    rng = np.random.default_rng(seed)
    y_true, p_a, p_b = map(np.asarray, (y_true, p_a, p_b))
    idx, diffs = np.arange(len(y_true)), []
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        diffs.append(f1_score(y_true[s], p_b[s], average="macro") - f1_score(y_true[s], p_a[s], average="macro"))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)

comparaciones = [
    ("4. TF-IDF + LR", "5. TF-IDF + léxico (original) + LR"),
    ("4. TF-IDF + LR", "6. TF-IDF + léxico (variante) + LR"),
    ("1. Léxico (original) + LR", "2. Léxico (variante) + LR"),
    ("0. Mayoritaria (baseline)", "2. Léxico (variante) + LR"),
]
filas = []
for a, b in comparaciones:
    d, lo, hi = paired_bootstrap(yte, preds[a], preds[b])
    filas.append({"A": a, "B": b, "Δ macro-F1 (B−A)": round(d, 4), "IC95% inf": round(lo, 4), "IC95% sup": round(hi, 4),
                  "IC excluye 0": (lo > 0) or (hi < 0)})
pd.DataFrame(filas)

NameError: name 'preds' is not defined

### 6.2 Errores por clase y matriz de confusión del mejor modelo (según CV)

In [ ]:
best_name = cv_table["CV macro-F1"].idxmax()
print("Mejor por CV:", best_name, "\n")
print(classification_report(yte, preds[best_name], labels=LABELS, zero_division=0, digits=3))

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay.from_predictions(yte, preds[best_name], labels=LABELS, ax=ax,
                                        cmap="Blues", colorbar=False)
ax.set_title(best_name, fontsize=9); plt.tight_layout(); plt.show()

### 6.3 Interpretación de los rasgos léxicos

Coeficientes de la regresión logística híbrida (TF-IDF + léxico, variante). Los rasgos están estandarizados, por lo que los coeficientes son comparables entre sí. Un coeficiente alto en una clase indica que ese rasgo empuja la predicción hacia ella, **manteniendo constante el resto**.

In [ ]:
hyb = fitted["6. TF-IDF + léxico (variante) + LR"]
clf = hyb.named_steps["clf"]
coef_lex = pd.DataFrame(clf.coef_[:, -len(FEATURE_NAMES):], index=clf.classes_, columns=FEATURE_NAMES).T
coef_lex["|max|"] = coef_lex.abs().max(axis=1)
coef_lex.sort_values("|max|", ascending=False).head(12).drop(columns="|max|").round(3)

In [ ]:
# Análisis de errores: ejemplos mal clasificados por el mejor modelo
err = pd.DataFrame({"texto": Xte["raw"], "real": yte, "predicho": preds[best_name]})
err = err[err["real"] != err["predicho"]]
print(f"Errores: {len(err)} de {len(Xte)} ({100*len(err)/len(Xte):.1f}%)")
err.sample(min(10, len(err)), random_state=SEED).reset_index(drop=True)

## 7. Cómo leer los resultados

Preguntas que los números de la sección 5 deben responder (con datos reales de TASS):

1. **¿Supera algún enfoque a la línea base mayoritaria?** Si el léxico solo no lo hace de forma clara, esos 29 rasgos no bastan por sí solos para detectar polaridad.
2. **Original vs. variante (filas 1 vs. 2, 5 vs. 6).** La diferencia mide el costo de los errores de preprocesamiento (emojis, hashtags y acentos perdidos).
3. **TF-IDF vs. híbrido (fila 4 vs. 6).** Si el intervalo bootstrap incluye 0, los rasgos léxicos no aportan información demostrable sobre TF-IDF con este tamaño de test.
4. **Regresión logística vs. Random Forest sobre léxico (fila 2 vs. 3).** Indica si hay interacciones no lineales entre rasgos.
5. **Errores por clase.** `NEU` es la clase minoritaria y su frontera con `NONE` es difusa; verifica en la matriz de confusión si concentra los errores.

**Limitaciones.**

- Los léxicos del curso no están validados como léxicos de sentimiento (ver sección 4).
- Una sola partición *train/test*; los intervalos bootstrap capturan la variabilidad de la muestra de test, no la de re-entrenamiento.
- Los tweets se preprocesan sin normalizar alargamientos (*buenoooo*), ironía ni negación.

**Extensiones para la actividad del curso.**

- Reemplazar TF-IDF por *embeddings* (Word2Vec, FastText) y luego por un modelo preentrenado en español (por ejemplo, BETO/RoBERTa-es) y comparar con este *baseline*.
- Corregir los problemas del diagnóstico en el código fuente y repetir los experimentos; documentar el efecto de cada corrección por separado (ablación).
- Añadir un léxico de polaridad validado para español y comparar contra `lexical_es`.

**Ejercicio de defensa oral.** Explica por qué el `TfidfVectorizer` se ajusta dentro de cada partición mientras que `LexicalVectorizer` puede calcularse antes de la validación cruzada.